# V4 CPU data and split audit

This self-contained audit validates the mounted xBD input using only the Python standard library. It does not train, tune thresholds, or submit another job.


In [ ]:
from __future__ import annotations

import json
import os
import struct
from collections import Counter
from pathlib import Path


INPUT_ROOT = Path('/kaggle/input')
OUTPUT_PATH = Path(os.environ.get('RV4_AUDIT_OUT', '/kaggle/working/rv4_cpu_audit.json'))
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.webp'}
LABEL_SUFFIXES = {'.json', '.geojson'}


def png_size(path: Path):
    try:
        with path.open('rb') as fh:
            if fh.read(8) != b'\x89PNG\r\n\x1a\n':
                return None
            length = struct.unpack('>I', fh.read(4))[0]
            chunk_type = fh.read(4)
            if chunk_type != b'IHDR' or length < 8:
                return None
            width, height = struct.unpack('>II', fh.read(8))
            return [width, height]
    except (OSError, struct.error):
        return None


def main() -> None:
    files = []
    suffix_counts = Counter()
    directory_counts = Counter()
    for root, dirs, names in os.walk(INPUT_ROOT):
        dirs.sort()
        for name in sorted(names):
            path = Path(root) / name
            try:
                rel = path.relative_to(INPUT_ROOT)
                size = path.stat().st_size
            except OSError:
                continue
            suffix = path.suffix.lower()
            suffix_counts[suffix or '<none>'] += 1
            directory_counts[rel.parts[0] if rel.parts else '<root>'] += 1
            files.append({'path': str(rel), 'bytes': size, 'suffix': suffix})

    image_files = [x for x in files if x['suffix'] in IMAGE_SUFFIXES]
    label_files = [x for x in files if x['suffix'] in LABEL_SUFFIXES]
    train_images = [x for x in image_files if '/train/' in f"/{x['path']}"]
    test_images = [x for x in image_files if '/test/' in f"/{x['path']}"]
    train_labels = [x for x in label_files if '/train/' in f"/{x['path']}"]

    label_samples = []
    label_diagnostics = []
    label_schema_ok = True
    for item in train_labels[:5]:
        path = INPUT_ROOT / item['path']
        try:
            raw_text = path.read_text()
            payload = json.loads(raw_text)
            diagnostic = {'path': item['path'], 'payload_type': type(payload).__name__}
            if isinstance(payload, dict):
                diagnostic['keys'] = sorted(payload.keys())[:20]
                diagnostic['features_type'] = type(payload.get('features')).__name__
            else:
                diagnostic['keys'] = []
                diagnostic['features_type'] = None
            diagnostic['prefix'] = raw_text[:240]
            label_diagnostics.append(diagnostic)
            features = payload.get('features')
            if isinstance(features, dict):
                for feature_key in ('xy', 'lng_lat', 'features'):
                    candidate = features.get(feature_key)
                    if isinstance(candidate, list):
                        features = candidate
                        break
            if not isinstance(features, list):
                label_schema_ok = False
                continue
            subtypes = []
            for feature in features[:10]:
                props = feature.get('properties', {}) if isinstance(feature, dict) else {}
                subtypes.append(props.get('subtype'))
            label_samples.append({'path': item['path'], 'feature_count': len(features), 'subtypes': sorted({x for x in subtypes if x})})
        except (OSError, json.JSONDecodeError, TypeError, AttributeError):
            label_schema_ok = False

    png_sample = None
    if image_files:
        sample_path = INPUT_ROOT / image_files[0]['path']
        png_sample = {'path': image_files[0]['path'], 'bytes': image_files[0]['bytes'], 'size': png_size(sample_path)}

    checks = {
        'input_root_exists': INPUT_ROOT.exists(),
        'input_has_files': bool(files),
        'has_images': bool(image_files),
        'has_train_images': bool(train_images),
        'has_test_images': bool(test_images),
        'has_train_labels': bool(train_labels),
        'label_schema_ok': label_schema_ok,
    }
    report = {
        'schema_version': 'rv4-cpu-audit-1',
        'status': 'PASS' if all(checks.values()) else 'FAIL',
        'checks': checks,
        'input_root': str(INPUT_ROOT),
        'mount_roots': dict(directory_counts),
        'file_count': len(files),
        'suffix_counts': dict(suffix_counts),
        'image_count': len(image_files),
        'label_count': len(label_files),
        'train_image_count': len(train_images),
        'test_image_count': len(test_images),
        'train_label_count': len(train_labels),
        'png_sample': png_sample,
        'label_samples': label_samples,
        'label_diagnostics': label_diagnostics,
    }
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
    print(json.dumps(report, indent=2, sort_keys=True))
    if report['status'] != 'PASS':
        raise SystemExit('V4 CPU audit failed')


if __name__ == '__main__':
    main()
